# Pole shape — data sanity check

Run this notebook from the **repository root** (so `data/images` resolves). In VS Code/Cursor: open the folder `pole-shape-classifer`, then run cells. Select the project interpreter: `.venv/bin/python`.

In [ ]:
%matplotlib inline

from pathlib import Path

import keras
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from keras import layers
from keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

ROOT = Path.cwd()
IMAGE_DIR = ROOT / "data" / "images"
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        f"Missing {IMAGE_DIR}. cd to the repo root or fix IMAGE_DIR."
    )

In [ ]:
# Split the data into train, val and test
batch_size = 32
seed = 1337
# Load 80% of the data for training
full_ds = keras.utils.image_dataset_from_directory(
    IMAGE_DIR,
    seed=seed,
    shuffle=True,
    image_size=(224, 224),
    batch_size=None
)

# Split into 80% train and 20% temp
train_ds, temp_ds = tf.keras.utils.split_dataset(full_ds, left_size=0.8, shuffle=True, seed=seed)

# Split the 20% temp into 10% val and 10% test (half of the remainder)
val_ds, test_ds = tf.keras.utils.split_dataset(temp_ds, left_size=0.5, shuffle=True, seed=seed)

# Batch AFTER splitting
train_ds = train_ds.batch(batch_size)
val_ds   = val_ds.batch(batch_size)
test_ds  = test_ds.batch(batch_size)


In [ ]:
class_names = full_ds.class_names
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    n = min(9, int(images.shape[0]))
    for i in range(n):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype("uint8"))
        lbl = int(labels[i])
        title = class_names[lbl] if lbl < len(class_names) else str(lbl)
        plt.title(title)
        plt.axis("off")
plt.tight_layout()
plt.show()

# Train baseline model - EfficientNetB0

In [ ]:
# 1. Load the base model with pre-trained ImageNet weights
# include_top=False removes the final 1000-class layer
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# 2. Freeze the base model (don't train its internal weights yet)
base_model.trainable = False

# 3. Add custom layers for your specific task (e.g., 2 classes: Cats vs. Dogs)
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.5)(x)  # Higher dropout rate to prevent overfitting
output = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# 4. Compile
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])



Basic model overfits since dataset is too small.

In [ ]:
# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10  # Start with 10 for transfer learning
)

In [ ]:
preds = model.predict(val_ds)


In [ ]:
# Assuming 'model' is trained and 'test_dataset' is ready

class_names = full_ds.class_names
for images, labels in val_ds.take(1):
    preds = model.predict(images)
    for i in range(9): # Display 9 images
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"T:{class_names[int(labels[i])]} \n P:{class_names[int(np.argmax(preds[i]))]}")
        plt.axis("off")
plt.show()

In [ ]:
y_true, y_pred = [], []
mis = []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0)
    pred = np.argmax(probs, axis=-1)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(pred.tolist())
    for i in range(len(labels)):
        yt, yp = int(labels[i]), int(pred[i])
        if yt != yp:
            mis.append((images[i].numpy(), yt, yp))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
print(f"misclassified: {np.sum(y_true != y_pred)} / {len(y_true)}")


In [ ]:
# save model
model.save("efficientnetb0_v1.keras")